<a href="https://colab.research.google.com/github/jjcodes999/flyrank-ml-repo/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jjcodes999/flyrank-ml-repo/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Answer:

**One row = one content page.** The warehouse's daily table stores one row per page per day (report_date × client × page). For my lane, I add up each page's days into one row per page.

**Table:** `fact_content_daily_performance` (the daily table), March 2026 partition only (`month=2026-03`). I use only its Search Console columns.

**Time window:** 1–31 March 2026, split into two halves:
- **Feature window:** 1–15 March. This is everything my model is allowed to know.
- **Outcome window:** 16–31 March. Used only to build the proxy label.
- **Decision moment:** the morning of 16 March. At that moment an editor can see the first half, but the second half has not happened yet.

I use March because it is a mid-panel month, away from the final month (June 2026), which I keep sealed as a test month.

**What I rank (proxy label):** `still_under` = 1 if a page's CTR in 16–31 March was still below the typical CTR for its position band in that same period, otherwise 0. As in ML-03, this is a proxy: it shows whether low CTR lasted, not whether editing the page would help.

**Filter:** only rows where `gsc_data_available IS TRUE`, and only pages with at least 100 impressions in the first half and 50 in the second half, so that CTR is not just noise.

**Where this is verified:** in Section 3. Query 1 checks the grain (one row per page per day, no duplicates), Query 2 checks the row count and that the dates run 1–31 March 2026, and Query 3 checks how many rows survive the `gsc_data_available IS TRUE` filter.

In [3]:
%pip -q install duckdb

import duckdb
from google.colab import userdata

# Read the token
HF_TOKEN = userdata.get("HF_TOKEN")

# Connect DuckDB to the warehouse on Hugging Face
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"

# the daily table
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

print("Connected. Working with: fact_content_daily_performance, month=2026-03")

Connected. Working with: fact_content_daily_performance, month=2026-03


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Answer:

**Features (clues), all from 1–15 March only:**
- `impressions_h1`: how many times the page was seen in search
- `ctr_h1`: clicks ÷ impressions × 100
- `position_h1`: average Google position, weighted by impressions
- `days_visible_h1`: on how many of the 15 days the page got at least one impression
- `missed_clicks_h1`: impressions × (typical CTR for its position band − its CTR) ÷ 100

**Label / proxy (the answer, never a clue):** `still_under`, built from `gsc_clicks`, `gsc_impressions` and `gsc_avg_position` in 16–31 March. Every second-half number (`clicks_h2`, `impressions_h2`, `ctr_h2`, `position_h2`) belongs here, not in the features.

**Context (for grouping, filtering and splitting only, never learned from):**
- `client_hash_id`, `content_hash_id`: scrambled IDs. Used to group days into pages and to test on clients the model has never seen.
- `report_date`: used only to split March into the two halves.
- `gsc_data_available`: used only as a filter (`IS TRUE`).

**Excluded (on purpose), with why:**
- **Any 16–31 March number as a feature.** This is my main deliberate exclusion: it is the outcome window, so using it would be seeing the future.
- **GA4 columns** (sessions, engagement, scroll, AI sessions): they measure what happens *after* someone clicks, so they cannot explain why people did not click. GA4 history also starts at different dates for different clients.
- **`gsc_sum_position`:** its exact meaning is not documented, so I compute position as an impressions-weighted average of `gsc_avg_position` instead.
- **June 2026 (the final month):** kept sealed as a future test month.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Answer:

Three queries on the March 2026 partition, each proving one claim from Sections 1–2:

**Query 1: grain.** Claim: one row of the daily table = one page on one day. Result: **0** duplicate (day, client, page) combinations, so the grain holds.

**Query 2: size and dates.** Claim: my slice is March 2026. Result: **9,841,378** rows covering **331,437** pages from **55** clients, running from **2026-03-01 to 2026-03-31** (31 days), with nothing outside March.

**Query 3: availability.** Claim: I keep only rows where search data is really present, filtering with `IS TRUE`. Result: **3,611,061** rows (**36.7%**) have `gsc_data_available` TRUE and survive the filter; **6,230,317** rows (63.3%) are FALSE and are dropped; **0** are empty in March.

What this tells me: almost two-thirds of the rows in this month carry no search data. Without the filter, those rows would add zeros that look like "nobody clicked" when really nothing was measured. The data dictionary warns that the flag can be empty in other months, so I filter with `IS TRUE` rather than `= TRUE` or `NOT FALSE`, which would mishandle empty values.

In [4]:
# QUERY 1 — GRAIN: is any page listed twice on the same day?
dupes = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n_rows
    FROM {MARCH}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("Duplicate (day, client, page) combinations found:", len(dupes))
display(dupes)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (day, client, page) combinations found: 0


,report_date,client_hash_id,content_hash_id,n_rows


In [5]:
# QUERY 2 — SIZE AND DATES: how big is my slice, and which days does it cover?
size = con.sql(f"""
    SELECT COUNT(*)                                                 AS total_rows,
           COUNT(DISTINCT client_hash_id || '|' || content_hash_id) AS pages,
           COUNT(DISTINCT client_hash_id)                           AS clients,
           MIN(report_date)                                         AS first_day,
           MAX(report_date)                                         AS last_day,
           COUNT(DISTINCT report_date)                              AS days
    FROM {MARCH}
""").df()

display(size)

,total_rows,pages,clients,first_day,last_day,days
0,9841378,331437,55,2026-03-01,2026-03-31,31


In [6]:
# QUERY 3 — AVAILABILITY: how many rows really have search data?
# The flag has THREE values (TRUE / FALSE / empty), so I filter with IS TRUE.
avail = con.sql(f"""
    SELECT COUNT(*)                                            AS all_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)  AS gsc_true,
           COUNT(*) FILTER (WHERE gsc_data_available IS FALSE) AS gsc_false,
           COUNT(*) FILTER (WHERE gsc_data_available IS NULL)  AS gsc_empty
    FROM {MARCH}
""").df()

avail["share_kept_pct"] = (avail["gsc_true"] / avail["all_rows"] * 100).round(1)
display(avail)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,all_rows,gsc_true,gsc_false,gsc_empty,share_kept_pct
0,9841378,3611061,6230317,0,36.7


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.